# 💾 Notebook 3 — Memory Bank (`core/memory_bank.py`)

**What this file does:** It is the "warehouse" of the defect detector. It saves and loads the compressed patch features, and builds a FAISS index on top of them for ultra-fast nearest-neighbour search.

**Why it exists:** At test time, for every patch in a test image we need to find the closest normal patch from training. Brute-force search on 200k+ vectors is too slow. FAISS (Facebook AI Similarity Search) makes this instant using GPU-accelerated indexing.

**How it fits:**
```
Train:  coreset.py → THIS FILE (save)
Test:   THIS FILE (load + index + search) → scoring.py
```

## 1 — Setup
Mount Google Drive, install FAISS, and configure logging.
This notebook loads the memory bank `.pt` files saved by Notebook 02.

In [ ]:
# ---- Mount Google Drive ----
from google.colab import drive
drive.mount('/content/drive')

# ---- Install FAISS ----
!pip install -q faiss-gpu

# ---- Imports ----
import os, logging
import numpy as np
import torch
import faiss
from typing import Tuple

# ---- Logging ----
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%H:%M:%S',
)
logger = logging.getLogger('memory_bank')

# ---- Paths ----
DRIVE_OUTPUT = '/content/drive/MyDrive/defects-detection-CV/outputs'
MEMORY_DIR   = os.path.join(DRIVE_OUTPUT, 'memory_banks')
FAISS_DIR    = os.path.join(DRIVE_OUTPUT, 'faiss_indices')
os.makedirs(FAISS_DIR, exist_ok=True)

CATEGORIES = ['bottle', 'carpet', 'screw']
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'

logger.info('Device: %s', DEVICE)

---
## 💾 `core/memory_bank.py` — Implementation
This file has **4 functions**, split into two groups:
- **Storage:** `save_memory_bank` and `load_memory_bank` (disk I/O)
- **Search:** `build_faiss_index` and `faiss_knn_search` (fast nearest-neighbour)

### Function 1 — `save_memory_bank`
Saves the memory bank tensor to a `.pt` file on disk.
Creates parent directories automatically if they don’t exist.
Used at the end of training to persist the compressed features.

In [ ]:
def save_memory_bank(
    memory_bank: torch.Tensor,
    save_path: str,
) -> None:
    """Serialize the memory bank tensor to disk.

    Args:
        memory_bank: (M, D) tensor of coreset patch features.
        save_path: File path (ending in .pt) to write to.
    """
    try:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        torch.save(memory_bank, save_path)
        logger.info('Saved memory bank %s → %s', memory_bank.shape, save_path)
    except Exception as exc:
        logger.error('Failed to save memory bank to %s: %s', save_path, exc)
        raise

### Function 2 — `load_memory_bank`
Reads a memory bank from disk and moves it to the specified device
(GPU or CPU). Used at the start of testing to restore saved features.

In [ ]:
def load_memory_bank(
    load_path: str,
    device: str,
) -> torch.Tensor:
    """Load a memory bank from disk.

    Args:
        load_path: Path to the .pt file.
        device: 'cuda' or 'cpu'.

    Returns:
        Memory bank tensor (M, D) on the requested device.
    """
    try:
        memory_bank = torch.load(load_path, map_location=device, weights_only=True)
        logger.info('Loaded memory bank %s from %s', memory_bank.shape, load_path)
        return memory_bank
    except FileNotFoundError:
        logger.error('Memory bank not found: %s', load_path)
        raise
    except Exception as exc:
        logger.error('Failed to load memory bank: %s', exc)
        raise

### Function 3 — `build_faiss_index`
Takes the memory bank and builds a **FAISS L2 index** on GPU.

Think of it like building a phone-book for feature vectors: instead of
scanning every entry one by one, FAISS organises them so lookups are
nearly instant. We use `IndexFlatL2` (exact search, no approximation)
because our coreset is small enough (~2k vectors) for exact search
to be fast.

Falls back to CPU automatically if GPU FAISS is unavailable.

In [ ]:
def build_faiss_index(
    memory_bank: torch.Tensor,
) -> faiss.Index:
    """Build a FAISS GPU index for fast nearest-neighbour search.

    Args:
        memory_bank: (M, D) tensor of coreset features.

    Returns:
        A populated FAISS index (GPU if available, else CPU).
    """
    vectors = memory_bank.detach().cpu().numpy().astype(np.float32)
    dim: int = vectors.shape[1]

    cpu_index = faiss.IndexFlatL2(dim)

    try:
        gpu_res = faiss.StandardGpuResources()
        gpu_index = faiss.index_cpu_to_gpu(gpu_res, 0, cpu_index)
        gpu_index.add(vectors)
        logger.info('FAISS GPU index built: %d vectors, dim=%d', gpu_index.ntotal, dim)
        return gpu_index
    except Exception as exc:
        logger.warning('FAISS GPU failed (%s), using CPU index.', exc)
        cpu_index.add(vectors)
        logger.info('FAISS CPU index built: %d vectors, dim=%d', cpu_index.ntotal, dim)
        return cpu_index

### Function 4 — `faiss_knn_search`
Given test patch features, search the index for the **k nearest neighbours**.
The returned distances tell us how "abnormal" each patch is —
high distance = the patch looks nothing like any normal training patch = defect.

In [ ]:
def faiss_knn_search(
    index: faiss.Index,
    query_features: torch.Tensor,
    k: int = 1,
) -> Tuple[np.ndarray, np.ndarray]:
    """Query the FAISS index for k nearest neighbours.

    Args:
        index: A populated FAISS index.
        query_features: (Q, D) tensor of test patch features.
        k: Number of neighbours to return.

    Returns:
        distances: (Q, k) numpy array of L2 distances.
        indices:   (Q, k) numpy array of neighbour indices.
    """
    queries = query_features.detach().cpu().numpy().astype(np.float32)
    distances, indices = index.search(queries, k)
    logger.debug('KNN: %d queries, k=%d, max_dist=%.4f', queries.shape[0], k, distances.max())
    return distances, indices

---
## ▶️ Run: Build FAISS Indices
Load each category’s memory bank from Notebook 02,
build a FAISS index, and verify it works with a quick sanity-check search.

In [ ]:
for category in CATEGORIES:
    bank_path = os.path.join(MEMORY_DIR, f'{category}_memory_bank.pt')

    # Load memory bank
    try:
        mb = load_memory_bank(bank_path, DEVICE)
    except FileNotFoundError:
        logger.error('[%s] Memory bank not found — run Notebook 02 first!', category)
        continue

    # Build FAISS index
    index = build_faiss_index(mb)

    # Sanity-check: search the first 5 vectors against themselves
    sample = mb[:5]
    dists, idxs = faiss_knn_search(index, sample, k=1)
    logger.info(
        '[%s] Sanity check — self-search distances (should be ~0): %s',
        category, dists.flatten().tolist(),
    )

    # Save the memory bank (confirm it's on Drive)
    save_memory_bank(mb.cpu(), bank_path)

logger.info('All FAISS indices built and verified!')

## ✅ Summary
At this point:
- Memory banks are confirmed saved on Drive.
- FAISS indices build successfully on GPU.
- Self-search sanity check passes (distances ≈ 0).

The FAISS index is built **in-memory** at test time (it’s instant for
~2k vectors), so we don’t need to save it to disk.

**Next:** The `train.py` notebook will tie feature_extractor + coreset +
memory_bank together into a single training pipeline.